[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiehou-lab/urban-ai/blob/main/notebooks/lab6_dss_ai_lab.ipynb)

# Lab 6: DSS AI Lab

**Duration:** ~1.5 hours
**TA lead:** Yura
**Course:** Urban AI — AI-Driven Decision Support for Real-World Urban Challenges (MSU AI-Ready Initiative)

## Learning goals
- Compare land-use / mobility scenarios across multiple outcome metrics.
- Compute a weighted composite score for each scenario under a given set of priorities.
- See how two defensible weighting philosophies can pick two different "winning" scenarios.
- Produce a Scenario Comparison Matrix (course deliverable 4).


## Before you start: Track A vs. Track B

Every Urban AI lab has two tracks. Both produce the **same artifact**: **Scenario Comparison Matrix (deliverable 4)**.

- **Track A — No code (default).** Run two scenarios in an approved no-code DSS platform. No installation, no Python required — use this track if you would rather click through a web tool.
- **Track B — Colab (this notebook).** Use a scenario engine: change parameters and compare outcomes side by side. You will run pre-written cells and change only the parameters marked `# ▶ CHANGE ME`. You will never need to write code from scratch.

Both tracks end with the same 4 reflection prompts (the last cell of this notebook).


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG = np.random.default_rng(6)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
print("Setup complete.")

## 2. Load the data: land-use / mobility scenario outcomes

> **Synthetic-but-realistic data.** The dataset below is generated in this notebook with a fixed
> random seed so the lab runs the same way for everyone, completely offline. It is built to look and
> behave like real urban data, but it is not real. To swap in real data for your own city, instructors
> can replace the data-generation cell with a download/load from a real source such as:
- Your regional Metropolitan Planning Organization (MPO) travel-demand model outputs
- City/county comprehensive plans, which often publish scenario comparison tables
- Real scenario-planning tools such as UrbanFootprint or Envision Tomorrow (institutional access required)


In [ ]:
base = {
    "Status Quo": dict(vmt_per_capita=22, transit_ridership=8000, new_housing=200,
                        displacement_risk=0.25, co2_tons=95000, cost_millions=15),
    "Transit-Oriented Development": dict(vmt_per_capita=14, transit_ridership=21000, new_housing=1800,
                                          displacement_risk=0.55, co2_tons=62000, cost_millions=180),
    "Highway Expansion": dict(vmt_per_capita=26, transit_ridership=6500, new_housing=350,
                               displacement_risk=0.35, co2_tons=118000, cost_millions=310),
}

rows = []
for name, vals in base.items():
    noisy_vals = {k: v * (1 + RNG.normal(0, 0.04)) for k, v in vals.items()}
    noisy_vals["scenario"] = name
    rows.append(noisy_vals)
scenario_outcomes = pd.DataFrame(rows).set_index("scenario").round(1)
scenario_outcomes

### Look at the raw outcome metrics
Each scenario trades off differently: TOD lowers driving and emissions but costs more and raises displacement risk; Highway Expansion adds almost no housing and raises emissions; Status Quo is cheap but delivers little of anything.

In [ ]:
fig_outcomes, axes = plt.subplots(2, 3, figsize=(14, 8))
metrics = list(scenario_outcomes.columns)
for ax, metric in zip(axes.flat, metrics):
    scenario_outcomes[metric].plot(kind="bar", ax=ax, color="tab:blue")
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Normalize and score under Weight Set A: Environment & Transit focus
Before combining metrics that are on totally different scales (tons of CO2 vs. dollars vs. a 0-1 risk index), we min-max normalize each into a 0-1 "benefit" score, then apply weights that favor low driving, high transit ridership, and low emissions.

In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)


norm = pd.DataFrame(index=scenario_outcomes.index)
norm["vmt_benefit"] = 1 - minmax(scenario_outcomes["vmt_per_capita"])
norm["transit_benefit"] = minmax(scenario_outcomes["transit_ridership"])
norm["housing_benefit"] = minmax(scenario_outcomes["new_housing"])
norm["displacement_benefit"] = 1 - minmax(scenario_outcomes["displacement_risk"])
norm["emissions_benefit"] = 1 - minmax(scenario_outcomes["co2_tons"])
norm["cost_benefit"] = 1 - minmax(scenario_outcomes["cost_millions"])

# ▶ CHANGE ME: Weight Set A -- an "Environment & Transit" priority set
weights_A = {"vmt_benefit": 0.15, "transit_benefit": 0.25, "housing_benefit": 0.15,
             "displacement_benefit": 0.15, "emissions_benefit": 0.25, "cost_benefit": 0.05}

score_A = (norm * pd.Series(weights_A)).sum(axis=1).sort_values(ascending=False)
winner_A = score_A.idxmax()

fig_scoreA, ax = plt.subplots(figsize=(7, 5))
score_A.plot(kind="bar", ax=ax, color="tab:green")
ax.set_title("Scenario Score under Weight Set A (Environment & Transit)")
ax.set_xlabel("Scenario")
ax.set_ylabel("Composite score (0-1)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print(f"Winner under Weight Set A: {winner_A}")
score_A.round(3)

### Responsible AI check: a different, equally defensible weight set picks a different winner
A city council member focused on near-term cost and displacement risk might reasonably weight the same metrics very differently. Watch what happens to the ranking.

In [ ]:
# ▶ CHANGE ME: Weight Set B -- a "Cost & Stability" priority set
weights_B = {"vmt_benefit": 0.05, "transit_benefit": 0.05, "housing_benefit": 0.10,
             "displacement_benefit": 0.30, "emissions_benefit": 0.10, "cost_benefit": 0.40}

score_B = (norm * pd.Series(weights_B)).sum(axis=1).sort_values(ascending=False)
winner_B = score_B.idxmax()

fig_scoreB, ax = plt.subplots(figsize=(7, 5))
score_B.plot(kind="bar", ax=ax, color="tab:orange")
ax.set_title("Scenario Score under Weight Set B (Cost & Stability)")
ax.set_xlabel("Scenario")
ax.set_ylabel("Composite score (0-1)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print(f"Winner under Weight Set A (Environment & Transit): {winner_A}")
print(f"Winner under Weight Set B (Cost & Stability):       {winner_B}")
if winner_A != winner_B:
    print("\nThe 'best' scenario depends entirely on whose priorities are encoded in the weights.")

## Experiment

Try changing the parameters marked `# ▶ CHANGE ME` in the next cell(s) and re-run. Specifically, try:

1. Edit `weights_B` to put more weight on `housing_benefit` -- does Transit-Oriented Development win instead?
2. Change `new_housing` for Transit-Oriented Development in the `base` dictionary above (re-run from there) and see how sensitive the winner is to that one assumption.
3. Design your own Weight Set C representing a specific stakeholder (e.g., a renter advocacy group) and see who wins.


## Artifact: Scenario Comparison Matrix (deliverable 4)

In [ ]:
scenario_comparison_matrix = scenario_outcomes.copy()
scenario_comparison_matrix["score_weight_set_A"] = score_A.round(3)
scenario_comparison_matrix["score_weight_set_B"] = score_B.round(3)
scenario_comparison_matrix["winner_under_A"] = scenario_comparison_matrix.index == winner_A
scenario_comparison_matrix["winner_under_B"] = scenario_comparison_matrix.index == winner_B

print(scenario_comparison_matrix)
scenario_comparison_matrix.to_csv("lab6_scenario_comparison_matrix.csv")
print("\nSaved artifact: lab6_scenario_comparison_matrix.csv (course deliverable 4)")

## Reflect (answer in your own words — this is part of your mini-task)

1. What did the tool assume?
2. Who is missing from this data?
3. What would change your recommendation?
4. What must a human verify before this is used?
